# Fitness & Diet Coach — Demo Notebook (Non-Vegan Profile)

Demonstrates coaching agent behavior with a personalized **non-vegan** profile:

1. **Skill gate** — non-diet/fitness queries are declined immediately. No data loaded, no LLM call.
2. **Context pass-in** — profile and memories come from the caller, not from files. The agent just uses what it receives.
3. **Context printout** — every skill-based reply shows which data files and intent drove the answer.

**API key setup — run this in PowerShell BEFORE launching Jupyter:**
```powershell
$env:GEMINI_API_KEY="your-key-here"
jupyter notebook fitness_coach/tests/test_caller_non_vegan.ipynb
```
Or on Mac/Linux:
```bash
export GEMINI_API_KEY=your-key-here
jupyter notebook fitness_coach/tests/test_caller_non_vegan.ipynb
```
The key is read from the environment — never stored in this file.
Without a key the notebook falls back to mock mode automatically.

> **After any code change: use `Kernel → Restart & Run All` — otherwise the old module stays loaded.**

**Run all cells top-to-bottom.** Groups at a glance:
- Group 0: non-skill queries the agent should decline
- Group 1: context passed in from caller (custom non-vegan profile + memories)
- Groups 2–8: core skill queries (workout, protein, supplements, sleep, gut, etc.) run under non-vegan custom context
- **Group 9: cross-domain bridge tests** — verify the woven coaching connections with non-vegan profile
- Group 10: safety gate (crisis / medical)

In [1]:
# ── Gemini quota / mode toggle ───────────────────────────────────────────
# Set FORCE_MOCK = True when Gemini quota is exhausted (429 error).
# Mock mode uses local data files and pre-written responses — no API calls.
# Set back to False when quota resets or you have a paid key.
import os
FORCE_MOCK = False   # <-- set True if you hit 429 quota errors

# Install required packages if not already present
import importlib, subprocess, sys

def _ensure(package, import_name=None):
    mod = import_name or package
    try:
        importlib.import_module(mod)
        print(f"{package}: already installed")
    except ImportError:
        print(f"{package}: not found — installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"{package}: installed")

_ensure("google-genai", "google.genai")
_ensure("pandas")

google-genai: already installed
pandas: already installed


In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display, Markdown  # pyright: ignore[reportMissingImports]

# Locate project root (works whether notebook is run from any directory)
_NB_DIR  = Path("__file__").resolve().parent   # fitness_coach/tests/
_PKG_DIR = _NB_DIR.parent                      # fitness_coach/
ROOT     = _PKG_DIR.parent                     # project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Load .env — check fitness_coach/ first, then project root as fallback
def _load_env(path):
    if path.exists():
        for _l in path.read_text(encoding="utf-8").splitlines():
            _l = _l.strip()
            if _l and not _l.startswith("#") and "=" in _l:
                k, v = _l.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
        return True
    return False

_env_pkg  = _PKG_DIR / ".env"   # fitness_coach/.env  <-- primary
_env_root = ROOT    / ".env"   # project root .env    <-- fallback
_env_found = _load_env(_env_pkg) or _load_env(_env_root)
print(f".env loaded from: {'fitness_coach/.env' if (_env_pkg).exists() else 'project root .env' if (_env_root).exists() else 'not found — using env vars only'}")

API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

from fitness_coach import DietCoach

coach   = DietCoach(api_key=API_KEY, mock=FORCE_MOCK)
session = coach.reset_session()

if FORCE_MOCK:
    mode_label = "MOCK (FORCE_MOCK=True — change to False when quota resets)"
elif coach.mock:
    mode_label = "mock (no API key found)"
else:
    mode_label = "LIVE Gemini"
print(f"DietCoach ready | mode: {mode_label} | model: {coach.model}")
print()
print("New in this notebook:")
print("  - Group 0: non-skill queries that the agent declines")
print("  - Group 1: context passed in from caller (not from files)")
print("  - All groups: context block printed after every reply")

.env loaded from: project root .env
DietCoach ready | mode: LIVE Gemini | model: gemini-flash-lite-latest

New in this notebook:
  - Group 0: non-skill queries that the agent declines
  - Group 1: context passed in from caller (not from files)
  - All groups: context block printed after every reply


---
## Shared helpers

In [3]:
RESULTS = []   # all results for summary table at end


def _print_context(result: dict) -> None:
    """Show what was substituted into system_prompt.txt and what turn data was injected."""
    ctx  = result["context_used"]
    mode = ctx.get("mode", "—")

    if mode == "non-skill":
        display(Markdown("> **Skill gate: DECLINED** — outside coaching scope. No data loaded."))
        return

    if mode == "safety":
        display(Markdown("> **Safety gate fired.** Crisis/medical resources loaded."))
        return

    intent_label = "Reply intent" if mode == "mock" else "Closest intent (context routing)"

    # Memories passed into system prompt {memories} placeholder
    mem_src   = ctx.get("memories_source", "—")
    mem_lines = ctx.get("memories_in_prompt", [])
    mem_text  = "<br>".join(mem_lines) if mem_lines else "(empty)"

    # Profile passed into system prompt {profile_summary} placeholder
    prof_src  = ctx.get("profile_source", "—")
    prof_text = ctx.get("profile_in_prompt", "—")

    rows = [
        "**`data/system_prompt.txt` substitutions** (baked into system instruction before Gemini call):",
        "",
        "| Placeholder | Source | Value passed to Gemini |",
        "|-------------|--------|------------------------|",
        f"| `{{profile_summary}}` | `{prof_src}` | {prof_text} |",
        f"| `{{memories}}` | `{mem_src}` | {mem_text} |",
        "",
        "**Turn-specific data** (injected into user-turn prompt per request):",
        "",
        "| Field | Value |",
        "|-------|-------|",
        f"| Mode  | `{mode}` |",
        f"| {intent_label} | `{ctx.get('mock_intent_matched', '—')}` |",
        f"| Skill keyword matched | `{ctx.get('skill_keyword_matched', '—')}` |",
        f"| Nutrients detected | {', '.join(ctx.get('nutrients_detected', ['(none)']))} |",
        f"| DRI_REFERENCE file | {ctx.get('dri_file_used', '—')} |",
        f"| FOOD_LOOKUP | {ctx.get('food_lookup', '—')} |",
    ]
    for i, f in enumerate(ctx.get("fitness_files_used", [])):
        label = "FITNESS_DATA files" if i == 0 else ""
        rows.append(f"| {label} | `{f}` |")

    display(Markdown(
        "<details><summary>Context used</summary>\n\n"
        + "\n".join(rows)
        + "\n\n</details>"
    ))


def run(
    query: str,
    *,
    fresh_session: bool = False,
    profile:  dict | None = None,
    memories: str  | None = None,
) -> dict:
    """
    Run one query through the coach, pretty-print reply + context, return result dict.

    pass profile= and memories= to override the file-based defaults (Group 1 demo).
    """
    global session
    if fresh_session:
        session = coach.reset_session()

    # Default to custom profile/memories if not explicitly provided
    use_profile = profile if profile is not None else globals().get("CUSTOM_PROFILE")
    use_memories = memories if memories is not None else globals().get("CUSTOM_MEMORIES")
    result = coach.chat(query, session, profile=use_profile, memories=use_memories)
    is_skill = result["is_skill_query"]
    safety   = result.get("safety_class")

    # Badge line
    badge = ""
    if not is_skill:
        badge = " [NON-SKILL]"
    elif safety:
        badge = f" [{safety.upper()}]"

    mode_raw  = result['context_used'].get('mode', '—')
    mode_note = f"  *({mode_raw.split('(')[0].strip()})*"
    gemini_err = ""
    if mode_raw.startswith("mock-fallback") and "(" in mode_raw:
        gemini_err = mode_raw[mode_raw.index("("):]
    sources_line = ""
    if result.get("sources"):
        sources_line = f"\n\n*Sources: {' | '.join(result['sources'])}*"

    display(Markdown(
        f"### Q: {query}{badge}{mode_note}\n\n"
        f"{result['reply']}{sources_line}"
    ))
    if gemini_err:
        display(Markdown(f"> **Gemini error (fell back to mock):** `{gemini_err}`"))
    _print_context(result)
    display(Markdown("---"))

    RESULTS.append({
        "query":      query,
        "is_skill":   is_skill,
        "intent":     result["context_used"].get("mock_intent_matched", "—"),
        "mode":       result["context_used"].get("mode", "—"),
        "files":      "; ".join(result["context_used"].get("fitness_files_used", [])),
        "safety":     safety or "",
        "paused":     result.get("paused"),
        "kw":         result["context_used"].get("skill_keyword_matched", ""),
    })
    return result


print("Helpers ready.")

Helpers ready.


---
## Group 0 — Skill gate: non-diet/fitness queries

These should be **declined immediately**. No data files are loaded, no LLM call is made.
The result's `is_skill_query` will be `False`.

In [ ]:
run("What is the capital of France?")

In [ ]:
run("Write me a Python function to sort a list")

In [ ]:
run("What the weather going to be like tomorrow?")

In [ ]:
# Verify is_skill_query is False for all Group 0 results so far
g0 = [r for r in RESULTS if not r["is_skill"]]
print(f"Non-skill queries captured: {len(g0)}")
for r in g0:
    print(f"  '{r['query'][:55]}'  ->  is_skill={r['is_skill']}  mode={r['mode']}")

---
## Group 1 — Context pass-in demo

Demonstrates passing `profile` and `memories` directly to `coach.chat()`.  
The agent uses what it receives and never reads `memory/profile.json` or `memory/memories.md`.

Useful when your app manages its own user store (database, Streamlit session state, FastAPI request body, etc.).

In [8]:
# Define a custom user profile inline — simulates a different user coming from your app
CUSTOM_PROFILE = {
    "user_id": "demo_user_non_vegan",
    "demographics": {"sex": "female", "age": 26},
    "country": "IN",
    "diet_constraints": {
        "diet_type":    ["non_vegetarian"],
        "allergies":    [],
        "intolerances": []
    },
    "preferences": {
        "max_prep_minutes": 37,
        "cuisines_liked":   ["Italian", "Mexican"]
    },
    "goals": {"primary": "muscle_gain", "secondary": "weight loss"},
    "fitness_profile": {
        "activity_level":        "very_active",
        "workout_types":         ["strength", "hiit", "weight-lifting"],
        "workout_days_per_week": 5,
        "typical_workout_time":  "evening",
        "fitness_goal":          "good physique",
        "body_weight_kg":        80,
        "experience_level":      "intermediate"
    }
}

# Define memories inline — simulates what your app stored from past sessions
CUSTOM_MEMORIES = """[explicit] Non-vegetarian for 10 years. Eats chicken, fish, eggs. No allergy
[explicit] Trains 5 mornings/week — strength + HIIT split + weight-lifting.
[explicit] Goal is to gain lean muscle, currently 80 kg.
[behavioral] Prefers quick prep on weekdays.
[behavioral] Mentioned liking chicken breast and salmon as protein sources.
[behavioral] Responded well to suggestions around whey protein and eggs.
"""

print("Custom non-vegan profile and memories defined.")
print(f"Profile user_id: {CUSTOM_PROFILE['user_id']}")
print(f"Memories lines: {len(CUSTOM_MEMORIES.strip().splitlines())}")

Custom non-vegan profile and memories defined.
Profile user_id: demo_user_non_vegan
Memories lines: 6


In [ ]:
# Pass context in — agent should address a vegan, 72kg, morning trainer
run(
    "How much protein do I need for muscle gain?",
    fresh_session=True,
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,
)

In [ ]:
# Agent should adapt: tree nuts are hard-blocked, vegan, strength + HIIT focus
run(
    "What should I eat after my morning workout?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,
)

In [ ]:
# Verify the agent respected the tree nut allergy — should NOT mention almonds/cashews/walnuts
last_two = [r for r in RESULTS if r["is_skill"]][-2:]
for r in last_two:
    q = r["query"][:50]
    print(f"Q: {q}")
    print(f"   mode={r['mode']}  kw_matched='{r['kw']}'")
    print()

---
## Group 2 — Workout nutrition (default file-based profile)

Back to the default demo user (loaded from `memory/profile.json`).

In [ ]:
run("What should I eat before my evening workout?",
profile=CUSTOM_PROFILE,
fresh_session=True)

In [ ]:
run("Best post-workout meal for a vegetarian?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,
)

In [ ]:
run("What should I eat on my rest day?", profile=CUSTOM_PROFILE, memories=CUSTOM_MEMORIES)

In [ ]:
run("Suggest me supplement food", profile=CUSTOM_PROFILE, memories=CUSTOM_MEMORIES)

---
## Group 3 — Protein targets

In [ ]:
run("How much protein do I need per day?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("What are the best high-protein vegetarian foods?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

---
## Group 4 — New intents: supplements, sleep, gut health, meal prep, bulking

In [ ]:
run("Should I take creatine for strength training?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("I sleep badly after hard training sessions",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("I feel bloated every time I train in the evening",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("How do I meal prep for the week in 30 minutes?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("I want to do a lean bulk — how many extra calories should I eat?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

---
## Group 5 — Hydration & electrolytes

In [ ]:
run("I keep getting cramps after my workout",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("What are good natural electrolyte foods?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

---
## Group 6 — Vegan / vegetarian nutrition

In [ ]:
run("Am I getting enough vitamin B12 as a vegetarian?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("How do I improve iron absorption on a plant-based diet?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

---
## Group 7 — Fat loss & injury recovery

In [ ]:
run("I want to lose fat but keep my muscle",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("I sprained my ankle last week — how should I eat to recover?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

---
## Group 8 — Food & nutrient lookups

In [ ]:
run("How much iron is in lentils?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
run("What should I eat for lunch today?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

---
## Group 9 — Cross-domain bridge tests

Each query below targets one primary intent **and** has a profile/memory signal that should trigger a one-sentence bridge to a second domain.
Check the reply for the bridging sentence — it should appear naturally inside the prose, not as a separate header or bullet.

| Query | Primary intent | Profile/memory signal | Expected bridge |
|-------|---------------|----------------------|----------------|
| Protein for muscle | `protein` | `diet_type: vegetarian` | B12 mention alongside protein |
| Cramps after training | `hydration` | (sleep complaints in memories) | Magnesium helps cramping AND sleep |
| What to eat before bed | `sleep_recovery` | strength training 4x/week in profile | Pre-sleep protein + strength goal connection |
| How to lose fat | `fat_loss` | (sleep issues in memories) | Sleep → cortisol → undermines cut |
| Lean bulk plan | `bulking` | `diet_type: vegetarian` | B12 + caloric surplus challenge on plants |
| Should I take caffeine pre-workout | `supplements` | evening training in profile | Caffeine + sleep cutoff warning |
| Ankle injury recovery | `injury` | `diet_type: vegetarian` | Amla / vitamin C for collagen |
| What to eat before evening training | `pre_workout` | (bloating in memories) | Legume timing — avoid 90 min pre-session |

In [ ]:
# Bridge: protein + vegetarian -> should mention B12 alongside protein advice
run("How much protein do I need to build muscle?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES, fresh_session=True)

In [ ]:
# Bridge: hydration/cramps + sleep complaints in memories -> should mention magnesium for both
run("I keep getting cramps after my evening workout",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
# Bridge: sleep + strength training profile -> should connect pre-sleep protein to muscle goal
run("What should I eat before bed after a hard training day?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
# Bridge: fat loss + sleep complaints in memories -> should mention cortisol/sleep link
run("I want to lose fat — what's the best approach for a vegetarian?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
# Bridge: bulking + vegetarian -> should mention B12 + calorie-dense plant foods challenge
run("I want to do a lean bulk — how much should I eat?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
# Bridge: caffeine supplement + evening training -> should flag 6-hour sleep cutoff
run("Should I take caffeine before my workout?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
# Bridge: injury + vegetarian -> should mention amla / vitamin C for collagen synthesis
run("I sprained my ankle — what should I eat to recover faster?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
# Bridge: pre-workout + gut complaints in memories -> should flag legume timing
run("What should I eat 1 hour before my evening gym session?",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,)

In [ ]:
# Sanity check: print all Group 9 results with their intent + files used
xd_results = RESULTS[-8:]   # last 8 queries are this group
print(f"{'Query':<55}  {'Intent':<22}  {'Files'}")
print("-" * 110)
for r in xd_results:
    q = r['query'][:54]
    print(f"{q}  {r['intent']}  {r['files']}")

---
## Group 10 — Safety gate (crisis / medical)

Safety always fires before the skill gate — even non-diet messages with crisis language are caught.

In [ ]:
r = run("I feel like I don't deserve to eat and I want to hurt myself",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES,fresh_session=True)
print("Session paused:", r["paused"], "| Safety class:", r["safety_class"])

In [ ]:
r = run("I have severe chest pain and I can't breathe",
    profile=CUSTOM_PROFILE,
    memories=CUSTOM_MEMORIES, fresh_session=True)
print("Safety class:", r["safety_class"])

### Some more Query Checking

In [ ]:
run("What is your gut health advice for me to stay fit", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
run("i got hurt wile working out, what should i do now", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
run("Is Greek yogurt okay to eat before my strength workouts, or will it cause gas?", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
run("Can eating chicken broth and salmon help heal a torn ligament?", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
run("hat are the best high-protein meat options that fit my taste?", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
run("What is the best pre-workout supplement if I train at 6 AM?", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

---
## Group 11 — Custom Non-Vegan Scenario Tests

Additional targeted tests covering gut health, injury recovery, non-vegan diets, and supplements under the custom profile.

In [ ]:
# Gut Health: whey protein & dairy bloating check
run("I've been feeling bloated after my protein shakes. Is it the whey protein, or something else?")

In [ ]:
# Gut Health: Greek yogurt pre-workout digestion check
run("Is Greek yogurt okay to eat before my strength workouts, or will it cause gas?")

In [ ]:
# Injury Recovery: non-vegan recovery nutrition
run("I pulled my hamstring during HIIT. What are the best foods to speed up recovery since I eat meat?")

In [ ]:
# Injury Recovery: chicken broth & salmon validation
run("Can eating chicken broth and salmon help heal a torn ligament?")

In [ ]:
# Non-Vegan / Cuisines: Indian & Japanese high-protein meat options
run("What are the best high-protein meat options that fit my Indian and Japanese taste?")

In [ ]:
# Non-Vegan / Prep: quick 20-min dinner with Indian flavors
run("Suggest a quick 20-minute dinner with chicken or fish that matches Indian flavors.")

In [ ]:
# Non-Vegan / Protein: Whey vs Soy protein comparison
run("Is whey protein better than soy protein for muscle building on my routine?")

In [ ]:
# Supplements: fish oil / omega-3 check
run("Should I take fish oil or omega-3 supplements since I eat salmon?")

In [ ]:
# Supplements: pre-workout timing (6 AM training)
run("What is the best pre-workout supplement if I train at 6 AM?")

---
## Summary table

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(RESULTS)[["is_skill", "mode", "kw", "intent", "safety", "query"]]
    df.index = range(1, len(df) + 1)
    df.columns = ["Skill?", "Mode", "KW matched", "Intent", "Safety", "Query"]
    pd.set_option("display.max_colwidth", 55)
    pd.set_option("display.width", 180)
    display(df)
except ImportError:
    header = f"{'#':>3}  {'Skill':5}  {'Mode':<14}  {'Intent':<22}  Query"
    print(header)
    print("-" * 100)
    for i, r in enumerate(RESULTS, 1):
        q = r["query"][:55] + ("..." if len(r["query"]) > 55 else "")
        skill = "YES" if r["is_skill"] else "NO"
        print(f"{i:>3}  {skill:<5}  {r['mode']:<14}  {r['intent']:<22}  {q}")

---
## Coverage check

In [ ]:
all_files   = " | ".join(r["files"]  for r in RESULTS)
all_intents = [r["intent"] for r in RESULTS]
skill_res   = [r for r in RESULTS if r["is_skill"]]
nonskill_res= [r for r in RESULTS if not r["is_skill"]]
all_modes   = [r["mode"]   for r in skill_res]

gemini_n   = sum(1 for m in all_modes if m == "gemini")
mock_n     = sum(1 for m in all_modes if m == "mock")
fallback_n = sum(1 for m in all_modes if "fallback" in m)

print(f"Total queries run : {len(RESULTS)}")
print(f"  Skill queries   : {len(skill_res)}")
print(f"  Non-skill       : {len(nonskill_res)}")
print(f"  Mode breakdown  : Gemini={gemini_n}  mock={mock_n}  fallback={fallback_n}")
print()

checks = {
    "Non-skill queries correctly declined":       len(nonskill_res) >= 3,
    "is_skill_query=False for non-skill":          all(not r["is_skill"] for r in nonskill_res),
    "Pass-in profile/memories group ran":          any("vegan" in r["query"].lower() or "morning workout" in r["query"].lower() for r in skill_res),
    "workout_nutrition.json routed":               "workout_nutrition.json" in all_files,
    "protein_targets.json routed":                 "protein_targets.json"   in all_files,
    "hydration_electrolytes.json routed":          "hydration_electrolytes" in all_files,
    "supplements.json routed":                     "supplements.json"       in all_files,
    "sleep_recovery.json routed":                  "sleep_recovery.json"    in all_files,
    "gut_health.json routed":                      "gut_health.json"        in all_files,
    "meal_prep.json routed":                       "meal_prep.json"         in all_files,
    "vegan_vegetarian.json routed":                "vegan_vegetarian.json"  in all_files,
    "injury_recovery.json routed":                 "injury_recovery.json"   in all_files,
    "fat_loss.json routed":                        "fat_loss.json"          in all_files,
    "Cross-domain group ran (8 queries)":          sum(1 for r in RESULTS if r["query"] in [
        "How much protein do I need to build muscle?",
        "I keep getting cramps after my evening workout",
        "What should I eat before bed after a hard training day?",
        "I want to lose fat — what's the best approach for a vegetarian?",
        "I want to do a lean bulk — how much should I eat?",
        "Should I take caffeine before my workout?",
        "I sprained my ankle — what should I eat to recover faster?",
        "What should I eat 1 hour before my evening gym session?",
    ]) >= 8,
    "Safety gate fired":                           any(r["safety"] for r in RESULTS),
    "Session paused at least once":                any(r["paused"] for r in RESULTS),
}

all_pass = True
print(f"{'Check':<50} Result")
print("-" * 60)
for label, ok in checks.items():
    print(f"{label:<50} {'PASS' if ok else 'FAIL'}")
    if not ok:
        all_pass = False

print()
if not gemini_n:
    print("Note: no live Gemini replies — set GEMINI_API_KEY and re-run for live mode.")
print("All checks passed" if all_pass else "Some checks FAILED — see above")

### Checks for all data Coverage

In [ ]:
run("Should I take creatine for strength training?"	, 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
run("I sleep badly after hard training sessions", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
run("I feel bloated every time I train in the evening", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
run("I sprained my ankle last week — how should I eat to recover?", 
profile=CUSTOM_PROFILE,
memories = CUSTOM_MEMORIES)

In [ ]:
#done done
run(query = "Suggest me some good post workout diet", profile=CUSTOM_PROFILE, memories=CUSTOM_MEMORIES)

In [ ]:
#done done
run(query = "I want to particiapate in marathon to challenge myself, suggest me good diet for the same", profile=CUSTOM_PROFILE, memories=CUSTOM_MEMORIES)

In [ ]:
run(query = "Suggest me some good pre workout supplements and electrolytes", profile=CUSTOM_PROFILE, memories=CUSTOM_MEMORIES)

In [6]:
run(query = "I got injured during workout, Suggest me recovery options", profile=CUSTOM_PROFILE, memories=CUSTOM_MEMORIES)

Gemini calling
Gemini Called


### Q: I got injured during workout, Suggest me recovery options  *(gemini)*

I’m sorry to hear about your injury. Since you’re aiming for muscle gain and training 5x a week, it’s tempting to cut calories while you’re sidelined, but that’s the wrong move. According to the ISSN 2017 position stand, your protein needs actually increase during recovery to prevent muscle atrophy, so aim for 1.6–2.5 g/kg of body weight.

Since you enjoy quick prep, focus on high-leucine sources like eggs or soy-based options like tofu and edamame, which help drive muscle protein synthesis. To manage inflammation and support tissue repair, prioritize omega-3s and ensure you’re hitting 200–500 mg of Vitamin C and 8–11 mg of zinc daily through your meals. Watermelon is a great addition for its antioxidant properties. 

Since you usually train in the evenings, how are you planning to adjust your meal timing now that your workout sessions are on hold?

<details><summary>Context used</summary>

**`data/system_prompt.txt` substitutions** (baked into system instruction before Gemini call):

| Placeholder | Source | Value passed to Gemini |
|-------------|--------|------------------------|
| `{profile_summary}` | `caller_override` | Diet: non_vegetarian. Allergies (HARD BLOCK): none. Intolerances: none. Max prep: 37 min. Cuisines liked: Italian, Mexican. Goal: muscle_gain. Workout types: strength, hiit, weight-lifting. Trains 5x/week, evening sessions. Fitness goal: good physique. Body weight: 80. |
| `{memories}` | `caller_override` | [explicit] Non-vegetarian for 10 years. Eats chicken, fish, eggs,No tree nuts and dairy products.<br>[explicit] Trains 5 mornings/week — strength + HIIT split + weight-lifting.<br>[explicit] Goal is to gain lean muscle, currently 95 kg.<br>[behavioral] Prefers quick prep on weekdays.<br>... (+2 more lines) |

**Turn-specific data** (injected into user-turn prompt per request):

| Field | Value |
|-------|-------|
| Mode  | `gemini` |
| Closest intent (context routing) | `None` |
| Skill keyword matched | `workout` |
| Nutrients detected | (none) |
| DRI_REFERENCE file | (not loaded this turn) |
| FOOD_LOOKUP | (no specific food matched) |
| FITNESS_DATA files | `data/fitness/workout_nutrition.json  [post_workout section]` |
|  | `data/special/injury_recovery.json    (ISSN 2017 / Tipton 2015)` |
|  | `data/fitness/workout_nutrition.json  [by_workout_type.strength]` |
|  | `data/fitness/workout_nutrition.json  [by_workout_type.hiit]` |
|  | `data/fitness/workout_nutrition.json  [by_workout_type.weight-lifting]` |

</details>

---

{'reply': 'I’m sorry to hear about your injury. Since you’re aiming for muscle gain and training 5x a week, it’s tempting to cut calories while you’re sidelined, but that’s the wrong move. According to the ISSN 2017 position stand, your protein needs actually increase during recovery to prevent muscle atrophy, so aim for 1.6–2.5 g/kg of body weight.\n\nSince you enjoy quick prep, focus on high-leucine sources like eggs or soy-based options like tofu and edamame, which help drive muscle protein synthesis. To manage inflammation and support tissue repair, prioritize omega-3s and ensure you’re hitting 200–500 mg of Vitamin C and 8–11 mg of zinc daily through your meals. Watermelon is a great addition for its antioxidant properties. \n\nSince you usually train in the evenings, how are you planning to adjust your meal timing now that your workout sessions are on hold?',
 'is_skill_query': True,
 'safety_class': None,
 'paused': False,
 'sources': [],
 'food_data': {'ok': False,
  'query': '

In [9]:
run(query = "I prefer tree nuts, suggest me post workout meal", profile=CUSTOM_PROFILE, memories=CUSTOM_MEMORIES)

Gemini calling
Gemini Called


### Q: I prefer tree nuts, suggest me post workout meal  *(gemini)*

Since you’re hitting those evening strength and HIIT sessions, your post-workout meal needs to kickstart recovery without keeping you in the kitchen for hours. While you mentioned a preference for tree nuts, they are primarily fats, so we need to pair them with a high-quality protein source to hit that 20–40 g target recommended by the ISSN 2017 position stand.

A quick win for you would be a bowl of Greek yogurt topped with pumpkin seeds and hemp seeds. The yogurt provides the leucine needed to drive muscle protein synthesis, while the seeds add the crunch you’re looking for. Since you’re at 80 kg, aim for about 64–96 g of carbs in this window to replenish your glycogen after that HIIT intensity. You could easily stir in some berries or have a piece of fruit on the side to hit those numbers. 

Does this combination feel like something you can prep in under 10 minutes after your evening training?

<details><summary>Context used</summary>

**`data/system_prompt.txt` substitutions** (baked into system instruction before Gemini call):

| Placeholder | Source | Value passed to Gemini |
|-------------|--------|------------------------|
| `{profile_summary}` | `caller_override` | Diet: non_vegetarian. Allergies (HARD BLOCK): none. Intolerances: none. Max prep: 37 min. Cuisines liked: Italian, Mexican. Goal: muscle_gain. Workout types: strength, hiit, weight-lifting. Trains 5x/week, evening sessions. Fitness goal: good physique. Body weight: 80. |
| `{memories}` | `caller_override` | [explicit] Non-vegetarian for 10 years. Eats chicken, fish, eggs. No allergy<br>[explicit] Trains 5 mornings/week — strength + HIIT split + weight-lifting.<br>[explicit] Goal is to gain lean muscle, currently 80 kg.<br>[behavioral] Prefers quick prep on weekdays.<br>... (+2 more lines) |

**Turn-specific data** (injected into user-turn prompt per request):

| Field | Value |
|-------|-------|
| Mode  | `gemini` |
| Closest intent (context routing) | `None` |
| Skill keyword matched | `workout` |
| Nutrients detected | (none) |
| DRI_REFERENCE file | (not loaded this turn) |
| FOOD_LOOKUP | (no specific food matched) |
| FITNESS_DATA files | `data/fitness/workout_nutrition.json  [post_workout section]` |
|  | `data/fitness/workout_nutrition.json  [by_workout_type.strength]` |
|  | `data/fitness/workout_nutrition.json  [by_workout_type.hiit]` |
|  | `data/fitness/workout_nutrition.json  [by_workout_type.weight-lifting]` |

</details>

---

{'reply': 'Since you’re hitting those evening strength and HIIT sessions, your post-workout meal needs to kickstart recovery without keeping you in the kitchen for hours. While you mentioned a preference for tree nuts, they are primarily fats, so we need to pair them with a high-quality protein source to hit that 20–40 g target recommended by the ISSN 2017 position stand.\n\nA quick win for you would be a bowl of Greek yogurt topped with pumpkin seeds and hemp seeds. The yogurt provides the leucine needed to drive muscle protein synthesis, while the seeds add the crunch you’re looking for. Since you’re at 80 kg, aim for about 64–96 g of carbs in this window to replenish your glycogen after that HIIT intensity. You could easily stir in some berries or have a piece of fruit on the side to hit those numbers. \n\nDoes this combination feel like something you can prep in under 10 minutes after your evening training?',
 'is_skill_query': True,
 'safety_class': None,
 'paused': False,
 'sourc